# Problem Set 1: Simple Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okuchap/GB656_2026_public/blob/main/problem-sets/02-01-template.ipynb)

In this notebook, you will study the relationship between car age and price in a synthetic used-car dataset.

## How to complete and submit the notebook

1. Open the template using the course Google Colab link.
2. Before editing, select **File > Save a copy in Drive**. Work only in the saved copy and rename it so the filename includes `PS1` and your name.
3. Run the cells from top to bottom. Complete every code and written-response **TODO** and replace each `...` in a code cell with your own code.
4. In Section 6, choose one additional car age and include it in the prediction workflow.
5. Before submitting, select **Runtime > Run all** and confirm that every requested output is visible and no cell reports an error. Save the notebook after the run finishes.
6. Click **Share**. Under **General access**, choose **Anyone with the link**, set the role to **Viewer**, and copy the sharing link.
7. Submit the link to your completed Drive copy as the **Website URL** in Canvas. Do not submit the original GitHub template link.
8. After the deadline, do not edit the submitted notebook unless the instructor asks you to resubmit.

Setup, display, and checking code is provided. You will complete selected data-preparation, model-fitting, and prediction lines, then interpret the results. Do not edit or delete instructor-provided cells, including the labeled result-check cells.

## 0. Setup

Google Colab already includes the packages used here, so no installation command is needed.

In [ ]:
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load and inspect the data

The instructor-provided helper below first looks for the CSV in a local course repository. If no local copy is available—as in a fresh Colab runtime—it loads the same file from the public course repository on GitHub. You do not need to upload the dataset or mount Google Drive.

In [ ]:
PUBLIC_REPOSITORY = "okuchap/GB656_2026_public"
PUBLIC_REVISION = "main"


def course_data_source(file_name):
    """Return a local course-data path when available, otherwise its public URL."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local_path = root / "data" / file_name
        if local_path.is_file():
            return local_path

    encoded_name = quote(file_name)
    return (
        "https://raw.githubusercontent.com/"
        f"{PUBLIC_REPOSITORY}/{PUBLIC_REVISION}/data/{encoded_name}"
    )


data_source = course_data_source("car_price_prediction.csv")
cars_raw = pd.read_csv(data_source)
source_location = (
    "local course repository" if isinstance(data_source, Path) else "public GitHub repository"
)
print(
    f"Loaded {cars_raw.shape[0]:,} rows and {cars_raw.shape[1]} columns "
    f"from the {source_location}."
)
cars_raw.head()

In [ ]:
cars_raw.info()

## 2. Prepare the analysis data

We use:

- outcome: `price`, interpreted as dollars for this assignment
- feature: `car_age`, measured in years

The dataset uses 2022 as its valuation year. We preserve that reference year for everyone:

$$\text{car\_age} = 2022 - \text{year}.$$

The column selection and renaming code is provided. Complete the marked line by adapting the vectorized feature-creation pattern from the lab.

In [ ]:
REFERENCE_YEAR = 2022

cars = (
    cars_raw[["Year", "Price"]]
    .dropna()
    .rename(columns={"Year": "year", "Price": "price"})
)

cars["car_age"] = ...  # TODO: create car age using REFERENCE_YEAR and the year column

cars[["year", "car_age", "price"]].head()

### Result check — do not edit

After completing the code above, run the next cell. It checks that `car_age` is numeric and follows the required 2022 reference-year formula. If it prints **Section 2 check passed**, continue to the summary table.

In [ ]:
assert pd.api.types.is_numeric_dtype(cars["car_age"]), "`car_age` should be numeric."
assert np.allclose(
    cars["car_age"], REFERENCE_YEAR - cars["year"]
), "Check the car-age formula."
print("Section 2 check passed.")

In [ ]:
summary_stats = cars[["year", "car_age", "price"]].describe().round(2)
summary_stats

### Question 2.1 — dataset age range

**TODO:** In one sentence, report the minimum and maximum car ages shown in the summary table.

**Your answer:**  

## 3. Visualize the relationship

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(cars["car_age"], cars["price"], alpha=0.5)
ax.set_xlabel("Car age (years)")
ax.set_ylabel("Synthetic car price ($)")
ax.set_title("Synthetic car price vs. car age")

plt.show()

### Question 3.1 — scatterplot

**TODO:** Describe the relationship. Is it positive, negative, or mostly flat? Is the direction consistent with depreciation intuition? Does age appear to explain every price difference?

**Your answer:**  

## 4. Fit the OLS regression

The model is

$$
\text{price}_i = \beta_0 + \beta_1\text{car\_age}_i + \epsilon_i.
$$

Complete the marked lines to:

1. define `y` as the `price` Series;
2. define `X` as a one-column DataFrame containing `car_age`;
3. add the intercept column with `sm.add_constant()`; and
4. fit the model with `sm.OLS(y, X).fit()`.

These steps directly adapt the model-input and OLS workflow practiced in the lab.

In [ ]:
y = ...  # TODO: select price as a Series
X = ...  # TODO: select car_age as a one-column DataFrame
X = ...  # TODO: add the intercept column

ols_model = ...  # TODO: specify and fit the OLS model

print(ols_model.summary())

### Result check — do not edit

After fitting the model, run the next cell. It checks the outcome, model columns, and fitted coefficient names. If it prints **Section 4 check passed**, continue to the coefficient output.

In [ ]:
assert y.equals(cars["price"]), "`y` should contain the price column."
assert X.columns.tolist() == ["const", "car_age"], (
    "`X` should contain const and car_age, in that order."
)
assert list(ols_model.params.index) == ["const", "car_age"], (
    "The fitted model should contain an intercept and a car_age slope."
)
print("Section 4 check passed.")

In [ ]:
intercept = ols_model.params["const"]
slope = ols_model.params["car_age"]

print(f"Estimated intercept: ${intercept:,.2f}")
print(f"Estimated slope:     ${slope:,.2f} per year")

In [ ]:
coef_table = pd.DataFrame(
    {
        "estimate": ols_model.params,
        "std_error": ols_model.bse,
    }
).rename(index={"const": "intercept"})

coef_table.round(4)

In [ ]:
print(f"Slope standard error: ${ols_model.bse['car_age']:,.2f}")
print(f"R-squared:             {ols_model.rsquared:.3f}")

## 5. Interpret the regression output

Use the full regression summary, the smaller coefficient table, and the printed estimates above.

### Question 5.1 — intercept

**TODO:** What is the estimated intercept? What would it mean here?

**Your answer:**  

### Question 5.2 — slope

**TODO:** What is the estimated slope on `car_age`? Interpret its sign, size, and units in context.

**Your answer:**  

### Question 5.3 — standard error

**TODO:** What is the standard error of the slope? What does a standard error measure?

**Your answer:**  

### Question 5.4 — $R^2$

**TODO:** What is $R^2$? What does 0.372 mean here?

**Your answer:**  

## 6. Make predictions

Make point predictions for ages 5, 10, 15, and 20, plus one additional whole-number age from 1 through 22. Complete the marked lines to build a prediction design matrix with the same columns as `X` and call the fitted model's `.predict()` method. Use `has_constant="add"` when adding the intercept, as practiced in the lab.

In [ ]:
required_ages = [5, 10, 15, 20]
MY_CAR_AGE = ...  # TODO: choose one whole-number age from 1 through 22

prediction_ages = required_ages + [MY_CAR_AGE]
predictions = pd.DataFrame({"car_age": prediction_ages})
predictions["model_year"] = REFERENCE_YEAR - predictions["car_age"]

prediction_X = ...  # TODO: add a constant to the car_age prediction DataFrame

predictions["predicted_price"] = ...  # TODO: call ols_model.predict()

required_predictions = predictions.iloc[: len(required_ages)].copy()
my_car = predictions.iloc[[-1]].copy()

required_predictions[["model_year", "car_age", "predicted_price"]].round(2)

### Result check — do not edit

After generating the predictions, run the next cell. It checks your chosen age, the prediction columns, and the predicted values. If it prints **Section 6 check passed**, continue to the plot.

In [ ]:
assert isinstance(MY_CAR_AGE, int) and 1 <= MY_CAR_AGE <= 22, (
    "Choose a whole-number age from 1 through 22."
)
assert prediction_X.columns.tolist() == X.columns.tolist(), (
    "Prediction columns must match the fitted-model columns."
)
assert np.allclose(
    predictions["predicted_price"],
    intercept + slope * predictions["car_age"],
), "Check the prediction call and design matrix."
print("Section 6 check passed.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

x_grid = np.linspace(cars["car_age"].min(), cars["car_age"].max(), 100)
y_grid = intercept + slope * x_grid

ax.scatter(cars["car_age"], cars["price"], alpha=0.25, label="Synthetic cars")
ax.plot(x_grid, y_grid, color="black", linewidth=2.5, label="Fitted line")
ax.scatter(
    required_predictions["car_age"],
    required_predictions["predicted_price"],
    color="red",
    marker="x",
    s=120,
    linewidths=3,
    label="Required predictions",
)
ax.set_xlabel("Car age (years)")
ax.set_ylabel("Synthetic car price ($)")
ax.set_title("Synthetic data, fitted line, and predictions")
ax.legend()

plt.show()

The next table reports the extra prediction for the whole-number age you chose above.

In [ ]:
my_car[["model_year", "car_age", "predicted_price"]].round(2)

### Question 6.1 — extra prediction

**TODO:** In one sentence, report and interpret the point prediction for your chosen age.

**Your answer:**  

## 7. Business summary

### Question 7.1 — recommendation

**TODO:** In 4–6 sentences, answer all three questions concisely:

1. What relationship did the model find?
2. How large is the estimated relationship?
3. Would you use this one-feature synthetic-data model as a real-world pricing tool? Explain why or why not, and name variables that could improve a final pricing model.

**Your answer:**  